In [1]:
import dataset_utils
import tensorflow as tf
import classifier_model
import tensorflow.keras as keras
import keras_tuner

In [2]:
BATCH_SIZE=32

In [3]:

num_ams_images , ams_ir_special_dataset = dataset_utils.get_dataset(list(range(1,13)), './data/ams_data/processed_images/patches/train/images', './data/ams_data/processed_images/patches/train/labels', dataset_name=None, dataset_source='AMS', dataset_type='classification')
ams_ir_special_dataset = ams_ir_special_dataset.shuffle(5000).batch(BATCH_SIZE,drop_remainder=True)
ams_ir_special_dataset = dataset_utils.balance_dataset(ams_ir_special_dataset, num_images=num_ams_images)
ams_ir_special_dataset = ams_ir_special_dataset.shuffle(5000).batch(BATCH_SIZE,drop_remainder=True)
print(num_ams_images)

4875


In [4]:
num_ams_images , ams_ir_special_test_dataset = dataset_utils.get_dataset(list(range(1,13)), './data/ams_data/processed_images/patches/test/images', './data/ams_data/processed_images/patches/test/labels',dataset_name=None, dataset_source='AMS', dataset_type='classification', test=True)
ams_ir_special_test_dataset = ams_ir_special_test_dataset.shuffle(5000).batch(BATCH_SIZE)
ams_ir_special_test_dataset = dataset_utils.balance_dataset(ams_ir_special_test_dataset, num_images=num_ams_images)
ams_ir_special_test_dataset = ams_ir_special_test_dataset.shuffle(5000).batch(BATCH_SIZE,drop_remainder=True)

In [5]:
ams_ir_special_dataset = ams_ir_special_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))
ams_ir_special_test_dataset = ams_ir_special_test_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))


In [6]:
def build_model(hp):
    model = classifier_model.build_model(12, 'simple', None)
    # Tune whether to use dropout.
    learning_rate = hp.Float("lr", min_value=1e-5, max_value=1e-3, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[
        keras.metrics.TruePositives(name='tp'),
      keras.metrics.FalsePositives(name='fp'),
      keras.metrics.TrueNegatives(name='tn'),
      keras.metrics.FalseNegatives(name='fn'), 
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.Precision(name='precision'),
      keras.metrics.Recall(name='recall'),
      keras.metrics.AUC(name='auc'),
      keras.metrics.AUC(name='prc', curve='PR'),
    ],
    )
    return model

In [7]:
tuner = keras_tuner.Hyperband(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=3,
    overwrite=True,
    directory="./data/hyperparamter_checkpoints",
    project_name="8-1-23trial",
)

In [8]:
tuner.search(ams_ir_special_dataset, 
                epochs=100,
                validation_data=ams_ir_special_test_dataset,
                callbacks=[keras.callbacks.EarlyStopping( monitor="val_loss",min_delta=5e-3,patience=10,verbose=1)],
                verbose=1
                )

Trial 10 Complete [00h 08m 06s]
val_accuracy: 0.6055555542310079

Best val_accuracy So Far: 0.7291666666666666
Total elapsed time: 01h 22m 17s
INFO:tensorflow:Oracle triggered exit
